# XGBoost

In [6]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# Read the training and test data
train_data = "/Users/alexandraantonica/Desktop/RW/augmented_data.csv"
train_df = pd.read_csv(train_data)
test_data = "/Users/alexandraantonica/Desktop/RW/normalized_orig_data.csv"
test_df = pd.read_csv(test_data)

# Split the data into features and target
X_train, y_train = train_df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']], train_df["categorical_output"]
X_test, y_test = test_df[['E', 'PEMi', 'Size KLOC', 'ACT_EFFORT']], test_df["categorical output"]

# Encode categorical output
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.fit_transform(y_test)

# Define the grid of hyperparameters to search over
param_grid = {
    'learning_rate': [0.001, 0.01, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.001, 0.01, 0.1, 1],
    'reg_lambda': [0, 0.001, 0.01, 0.1, 1]
}

# Initialize XGBoost classifier
xgb_classifier = XGBClassifier(objective='multi:softmax', num_class=len(set(y_train)))

# Create GridSearchCV object
grid_search = GridSearchCV(xgb_classifier, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Perform grid search on training data
grid_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_

# Initialize XGBoost classifier with best hyperparameters
best_xgb_classifier = XGBClassifier(objective='multi:softmax', num_class=len(set(y_train)), **best_params)

# Train the model with best hyperparameters on the entire training data
best_xgb_classifier.fit(X_train, y_train)

# Making predictions on the training and test sets with best hyperparameters
y_train_pred_best = best_xgb_classifier.predict(X_train)
y_test_pred_best = best_xgb_classifier.predict(X_test)

# Calculate accuracy and recall with best hyperparameters
train_accuracy_best = accuracy_score(y_train, y_train_pred_best)
test_accuracy_best = accuracy_score(y_test, y_test_pred_best)
train_recall_best = recall_score(y_train, y_train_pred_best, average='macro')
test_recall_best = recall_score(y_test, y_test_pred_best, average='macro')

# Compute confusion matrix with best hyperparameters
conf_matrix_best = confusion_matrix(y_test, y_test_pred_best)

print("Best Hyperparameters:", best_params)
print(f"Training Accuracy with best hyperparameters: {train_accuracy_best:.3f}")
print(f"Test Accuracy with best hyperparameters: {test_accuracy_best:.3f}")
print(f"Training Recall with best hyperparameters: {train_recall_best:.3f}")
print(f"Testing Recall with best hyperparameters: {test_recall_best:.3f}")
print("Confusion matrix with best hyperparameters:")
print(conf_matrix_best)


Best Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1, 'reg_alpha': 1, 'reg_lambda': 0.1, 'subsample': 1.0}
Training Accuracy with best hyperparameters: 0.624
Test Accuracy with best hyperparameters: 0.803
Training Recall with best hyperparameters: 0.486
Testing Recall with best hyperparameters: 0.736
Confusion matrix with best hyperparameters:
[[10  1  0]
 [ 3 45  2]
 [ 0  9  6]]
